In [2]:
#Checking to see if my data files are here
import os
print("CWD:", os.getcwd())
print(os.listdir())

CWD: /Users/Maya/Desktop/species
['taxa.csv', '.DS_Store', 'species_train_extra.npz', 'Final! Climate 2.5 minutes path extracted as averages.ipynb', 'species_test.npz', 'explore_species_data.py', 'tmax_annual.npy', 'wc2.1_2.5m_tmin', 'species.miniproc with tempreature.ipynb', '.ipynb_checkpoints', 'prec_annual.npy', 'species_train.npz', 'wc2.1_2.5m_tmax', 'tmin_annual.npy', 'wc2.1_2.5m_prec']


In [3]:
import numpy as np
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [4]:
TRAIN_LOCS_KEY = 'train_locs'
TRAIN_IDS_KEY = 'train_ids'
TAXON_IDS_KEY = 'taxon_ids'
TAXON_NAME_KEY = 'taxon_names'

Reading the file:

In [5]:
filepath = os.path.join(os.getcwd(), 'species_train.npz')
data = np.load(filepath, allow_pickle=True)
train_locs = data[TRAIN_LOCS_KEY]
train_ids = data[TRAIN_IDS_KEY]
taxon_ids = data[TAXON_IDS_KEY]
taxon_names = data[TAXON_NAME_KEY]

Mapping the taxon ids to taxon latin names: 

In [6]:
species_ids_names = dict(zip(data['taxon_ids'], data['taxon_names']))  # latin names of species 

Create pandas Dataframe from the data: 

In [7]:
df = pd.DataFrame({
    'latitude': train_locs[:, 0],
    'longitude': train_locs[:, 1], 
    'taxon_id': data['train_ids']
})
df['taxon_name'] = [species_ids_names[id] for id in data['train_ids'].astype(int)]
df.head()

,latitude,longitude,taxon_id,taxon_name
0,-18.286728,143.481247,31529,Lophognathus gilberti
1,-13.099798,130.783646,31529,Lophognathus gilberti
2,-13.965274,131.695145,31529,Lophognathus gilberti
3,-12.853950,132.800507,31529,Lophognathus gilberti
4,-12.196790,134.279327,31529,Lophognathus gilberti


In [8]:
df.shape

(272037, 4)

Data Cleanining: 

<small>1. Check for missing or invalid coordinates:</small>

In [9]:
df = df.dropna(subset=['latitude', 'longitude'])
df = df[(df['latitude'].between(-90, 90)) & (df['longitude'].between(-180, 180))]
df.shape

(272037, 4)

<small>2. Remove any duplicates or nearly duplicates (observations that are extremely close):</small>

In [10]:
df['lat_rounded'] = df['latitude'].round(5)
df['lon_rounded'] = df['longitude'].round(5)

In [11]:
df = df.drop_duplicates(subset=['lat_rounded', 'lon_rounded'])
df.shape

(237977, 6)

<small>4. Validate species IDs: </small>

In [12]:
df['taxon_id'].isna().sum()

np.int64(0)

<small>5. Convert to categorical labels:</small>

In [13]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['taxon_id'])
df.head()

,latitude,longitude,taxon_id,taxon_name,lat_rounded,lon_rounded,label
0,-18.286728,143.481247,31529,Lophognathus gilberti,-18.28673,143.481247,278
1,-13.099798,130.783646,31529,Lophognathus gilberti,-13.09980,130.783646,278
2,-13.965274,131.695145,31529,Lophognathus gilberti,-13.96527,131.695145,278
3,-12.853950,132.800507,31529,Lophognathus gilberti,-12.85395,132.800507,278
4,-12.196790,134.279327,31529,Lophognathus gilberti,-12.19679,134.279327,278


<small>6. Only keep birds:</small>

<small>Note: Only run the next 2 blocks one time as they take a few seconds:</small>

In [14]:
taxa = pd.read_csv('taxa.csv')
birds = taxa[taxa['class'] == 'Aves']
bird_taxon_ids = set(birds['id'])
len(bird_taxon_ids)

32140

In [15]:
df = df[df['taxon_id'].isin(bird_taxon_ids)].copy()
df.shape

(151391, 7)

<small>Append the temprature data</small>

In [16]:
# Add the temprature data
# === Climate rasters: load 12-month stacks for Tmin/Tmax/Prec ===
import rasterio
import numpy as np
import os

base_dir = os.getcwd()  # assumes folders are in the same directory as this notebook

def load_stack(folder_name):
    folder = os.path.join(base_dir, folder_name)
    files = sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".tif")])
    if len(files) != 12:
        raise RuntimeError(f"Expected 12 GeoTIFFs in {folder_name}, found {len(files)}.")
    layers = []
    transform = None
    for f in files:
        with rasterio.open(f) as src:
            layers.append(src.read(1))
            transform = src.transform  # all months share the same transform
    return np.stack(layers), transform  # shape: (12, H, W), and affine transform

tmin, transform = load_stack("wc2.1_2.5m_tmin")
tmax, _         = load_stack("wc2.1_2.5m_tmax")
prec, _         = load_stack("wc2.1_2.5m_prec")

print("Stacks loaded. Shapes:",
      "tmin", tmin.shape, "tmax", tmax.shape, "prec", prec.shape)
print("Transform:", transform)


Stacks loaded. Shapes: tmin (12, 4320, 8640) tmax (12, 4320, 8640) prec (12, 4320, 8640)
Transform: | 0.04, 0.00,-180.00|
| 0.00,-0.04, 90.00|
| 0.00, 0.00, 1.00|


In [17]:
#Cleaning the data again as the precipitation values are very large, so I'll normalize them
from rasterio.transform import rowcol
import numpy as np

def get_climate_for_points(df, transform, tmin, tmax, prec, lat_col="latitude", lon_col="longitude"):
    # Convert lat/lon → raster indices
    rows, cols = rowcol(transform, df[lon_col].values, df[lat_col].values)
    rows = np.clip(rows, 0, tmin.shape[1]-1)
    cols = np.clip(cols, 0, tmin.shape[2]-1)

    # Helper: clean up weird values (NoData etc.)
    def clean(arr):
        arr = arr.astype(float)
        arr[arr > 1e4] = np.nan  # remove unrealistic large values
        return arr

    tmin = clean(tmin)
    tmax = clean(tmax)
    prec = clean(prec)

    # Extract values and average across 12 months safely
    tmin_mean = np.nanmean(tmin[:, rows, cols], axis=0)
    tmax_mean = np.nanmean(tmax[:, rows, cols], axis=0)
    prec_mean = np.nanmean(prec[:, rows, cols], axis=0)

    return tmin_mean, tmax_mean, prec_mean



In [18]:
tmin_avg, tmax_avg, prec_avg = get_climate_for_points(df, transform, tmin, tmax, prec)


In [19]:
df["Tmin_avg"] = tmin_avg
df["Tmax_avg"] = tmax_avg
df["Prec_avg"] = prec_avg

df[["latitude","longitude","Tmin_avg","Tmax_avg","Prec_avg"]].head()


,latitude,longitude,Tmin_avg,Tmax_avg,Prec_avg
53,21.086105,-86.852867,20.967000,31.436000,103.416667
54,19.186003,-96.199600,19.327333,31.831333,117.833333
55,17.538877,-89.113724,19.496333,30.576333,125.583333
56,20.648556,-105.220955,19.674123,31.835527,87.750000
57,18.409698,-95.096657,20.515000,29.086000,156.750000


In [20]:
out_path = "bird_species_rerun_with_averaged_climate.csv"
df.to_csv(out_path, index=False)
out_path, df.shape


('bird_species_rerun_with_averaged_climate.csv', (151391, 10))

<small>8. Split the data to x and y</small>

In [21]:
x_data = df.drop(columns=['taxon_id', 'taxon_name', 'lat_rounded', 'lon_rounded', 'label'])
y_data = df['label']

In [22]:
#Casts data to float so it can hold NaN values.

#Masks any absurd values (above 10 000 °C or mm, or below −10 000) as NaN.

#Uses np.nanmean, which quietly ignores those cells while averaging.
import numpy as np

def safe_mean(arr):
    arr = arr.astype(float)
    arr[(arr > 1e4) | (arr < -1e4)] = np.nan   # mask nodata outliers
    return np.nanmean(arr, axis=0)

tmin_annual = safe_mean(tmin)
tmax_annual = safe_mean(tmax)
prec_annual = safe_mean(prec)

np.save("tmin_annual.npy", tmin_annual)
np.save("tmax_annual.npy", tmax_annual)
np.save("prec_annual.npy", prec_annual)

print("Saved annual mean rasters:")
print("tmin_annual:", tmin_annual.shape, "tmax_annual:", tmax_annual.shape, "prec_annual:", prec_annual.shape)


/var/folders/h0/0kms8dqj2q3_fm6zm8kf0kdw0000gq/T/ipykernel_95495/4116940445.py:11: RuntimeWarning: Mean of empty slice
  return np.nanmean(arr, axis=0)


Saved annual mean rasters:
tmin_annual: (4320, 8640) tmax_annual: (4320, 8640) prec_annual: (4320, 8640)


<small>9. Split to train and validation sets</small>

Exploratory Data Analysis:

In [23]:
# Hajer

Models: